# TOMAS ARC-AGI-3 Solver v2.4 — Kaggle Notebook

Taiyi Mutual-Play (TOMAS) framework for ARC-AGI-3 video reasoning competition.

**v2.4 Features:**
- psi-Gate Semantic Gating (enabled by default)
- AEGIS Evolution Engine (optional)
- Causal DSL Prior (optional)
- RHAE Evaluation Framework
- Numba JIT + CUDA GPU acceleration

## 1. Environment Setup

In [ ]:
import sys
import os

# Install dependencies
!pip install -q numpy scipy networkx pyyaml loguru tqdm httpx numba

# Check GPU
try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
except ImportError:
    print('PyTorch not available, using CPU-only mode')

# Check Numba
try:
    from numba import cuda, njit
    print(f'Numba CUDA available: {cuda.is_available()}')
except ImportError:
    print('Numba not available, install with: pip install numba')

## 2. Load TOMAS Solver v2.4

In [ ]:
# Copy source code or clone repo
# For Kaggle, upload the tomas-arc3-solver as a dataset
sys.path.insert(0, '/kaggle/input/tomas-arc3-solver')

from src.utils.config import ConfigLoader
from src.utils.logger import get_auditor
from src.solver.tomas_solver import TOMASSolver

# Load config
config = ConfigLoader.load('/kaggle/input/tomas-arc3-solver/config/default.yaml')

# Override for Kaggle environment
config['kaggle']['input_dir'] = '/kaggle/input/arc-agi-3'
config['kaggle']['output_dir'] = '/kaggle/working'
config['vl_api']['available'] = False  # No external API in Kaggle
config['gpu']['max_vram_gb'] = 16  # T4 has 16GB

# === v2.4 Feature Configuration ===
# psi-Gate: Enabled by default for better fusion quality
config['psi_gate']['enabled'] = True
config['psi_gate']['use_default_anchors'] = True
config['psi_gate']['tolerance_decay_rate'] = 0.05

# AEGIS: Enable for evolution search (slower but higher accuracy)
config['aegis']['enabled'] = True
config['aegis']['max_generations'] = 3
config['aegis']['population_size'] = 20
config['aegis']['mutation_rate'] = 0.15

# Causal DSL Prior: Enable for smarter candidate ordering
config['causal_prior']['enabled'] = True

# RHAE Evaluation (for local scoring)
config['eval']['enabled'] = False  # Disable in competition, enable for testing

print('=== TOMAS v2.4 Configuration ===')
print(f'  psi-Gate: {"ON" if config["psi_gate"]["enabled"] else "OFF"}')
print(f'  AEGIS: {"ON" if config["aegis"]["enabled"] else "OFF"}')
print(f'  Causal Prior: {"ON" if config["causal_prior"]["enabled"] else "OFF"}')
print(f'  RHAE Eval: {"ON" if config["eval"]["enabled"] else "OFF"}')

## 3. Initialize Solver

In [ ]:
solver = TOMASSolver(config)
print('Solver initialized with v2.4 features.')
print(f'  Mode: {solver.mode}')
print(f'  Search max_depth: {solver.searcher.max_depth}')
print(f'  Search mdl_threshold: {solver.searcher.mdl_threshold}')
print(f'  CUDA backend: {solver.searcher._cuda_backend}')
print(f'  Pruning optimizer: {"active" if solver.searcher.pruning else "disabled"}')
if hasattr(solver, 'psi_gate') and solver.psi_gate:
    print(f'  psi-Gate: active ({len(solver.psi_gate.anchors)} anchors)')

## 4. Load Competition Data

In [ ]:
import json
from pathlib import Path

input_dir = Path('/kaggle/input/arc-agi-3')
task_files = sorted(input_dir.glob('**/*.json'))
print(f'Found {len(task_files)} task files')

# Preview first task
if task_files:
    with open(task_files[0], 'r') as f:
        first_task = json.load(f)
    print(f'Sample task: {task_files[0].name}')
    print(f'  Train pairs: {len(first_task.get("train", []))}')
    print(f'  Test pairs: {len(first_task.get("test", []))}')

## 5. Solve All Tasks

In [ ]:
from tqdm import tqdm
import time
import numpy as np

results = {}
stats = {
    'total': 0, 'solved': 0, 'failed': 0,
    'total_time': 0, 'total_candidates': 0,
}
start_time = time.time()

for tf in tqdm(task_files, desc='Solving'):
    try:
        with open(tf, 'r') as f:
            task_data = json.load(f)
        task_data['task_id'] = tf.stem
        
        stats['total'] += 1
        
        # Auto-select mode based on time budget
        remaining_time = max(10.0, 80.0 - (time.time() - start_time) / max(stats['total'], 1))
        mode = solver.auto_select_mode(remaining_time, len(task_data.get('train', [])))
        
        result = solver.solve(task_data, mode=mode)
        results[tf.stem] = result
        
        if result is not None:
            stats['solved'] += 1
        else:
            stats['failed'] += 1
        
        # Library Learning feedback
        solver._post_solve_learning(result)
        
    except Exception as e:
        print(f'Error solving {tf.name}: {e}')
        results[tf.stem] = None
        stats['failed'] += 1

elapsed = time.time() - start_time
stats['total_time'] = elapsed

print(f'\n=== Solving Complete ===')
print(f'  Total: {stats["total"]}')
print(f'  Solved: {stats["solved"]}')
print(f'  Failed: {stats["failed"]}')
print(f'  Time: {elapsed:.1f}s ({elapsed/max(stats["total"],1):.1f}s/task)')

## 6. AEGIS Evolution (Optional - for difficult tasks)

In [ ]:
# Re-solve failed tasks with AEGIS evolution engine
if config['aegis']['enabled'] and stats['failed'] > 0:
    from src.solver.aegis_evolver import AEGISEvolver
    
    aegis = AEGISEvolver(config.get('aegis', {}))
    
    for task_id, result in list(results.items()):
        if result is None:
            print(f'Re-trying {task_id} with AEGIS evolution...')
            try:
                # Find the task file
                task_file = next(tf for tf in task_files if tf.stem == task_id)
                with open(task_file, 'r') as f:
                    task_data = json.load(f)
                
                # Run AEGIS evolution
                evolved_result = aegis.evolve_search(
                    task_data, solver.searcher, max_generations=3
                )
                if evolved_result:
                    results[task_id] = evolved_result
                    stats['solved'] += 1
                    stats['failed'] -= 1
                    print(f'  -> Solved via AEGIS!')
                else:
                    print(f'  -> Still failed')
            except Exception as e:
                print(f'  -> AEGIS error: {e}')
else:
    print('AEGIS disabled or no failed tasks to retry.')

## 7. Generate Submission

In [ ]:
from src.utils.kaggle_format import KaggleFormatAdapter

adapter = KaggleFormatAdapter()
submission = {}
for task_id, pred in results.items():
    if pred is not None:
        submission[task_id] = pred

output_path = '/kaggle/working/submission.json'
with open(output_path, 'w') as f:
    json.dump(submission, f)

print(f'Submission saved to {output_path}')
print(f'Tasks submitted: {len(submission)} / {stats["total"]}')
print(f'Submission rate: {len(submission)/max(stats["total"],1)*100:.1f}%')

## 8. Verify Output Format

In [ ]:
valid = adapter.validate_output(submission)
print(f'Output valid: {valid}')
print(f'Submission size: {os.path.getsize(output_path)} bytes')

# Summary statistics
print(f'\n=== Final Summary ===')
print(f'  Total tasks: {stats["total"]}')
print(f'  Submitted: {len(submission)}')
print(f'  Success rate: {len(submission)/max(stats["total"],1)*100:.1f}%')
print(f'  Total time: {stats["total_time"]:.1f}s')
print(f'  Avg time/task: {stats["total_time"]/max(stats["total"],1):.1f}s')
print(f'  psi-Gate: {"ON" if config["psi_gate"]["enabled"] else "OFF"}')
print(f'  AEGIS: {"ON" if config["aegis"]["enabled"] else "OFF"}')
print(f'  Causal Prior: {"ON" if config["causal_prior"]["enabled"] else "OFF"}')

## 9. Post-Submission Analysis

In [ ]:
# Analyze pruning effectiveness
if hasattr(solver.searcher, 'pruning') and solver.searcher.pruning:
    pruning_stats = solver.searcher.pruning.stats
    print('=== Pruning Statistics ===')
    for strategy, count in pruning_stats.items():
        print(f'  {strategy}: {count}')

# Analyze ENPV decisions
if hasattr(solver.searcher, 'enpv'):
    print(f'\n=== ENPV Decision ===')
    print(f'  Early terminated: {getattr(solver.searcher.enpv, "_early_terminated", "N/A")}')

# Save run log
log_data = {
    'version': '2.4.0',
    'stats': stats,
    'config': {
        'psi_gate': config['psi_gate']['enabled'],
        'aegis': config['aegis']['enabled'],
        'causal_prior': config['causal_prior']['enabled'],
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open('/kaggle/working/run_log.json', 'w') as f:
    json.dump(log_data, f, indent=2)
print(f'\nRun log saved to /kaggle/working/run_log.json')